In [1]:
import pandas as pd 
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os 

In [2]:
load_dotenv()

USER = os.getenv('DB_USER')
PASS = os.getenv('DB_PASS')
HOST = os.getenv('DB_HOST')
PORT = os.getenv('DB_PORT')
NAME = os.getenv('DB_NAME')

engine = create_engine(f'mysql+pymysql://{USER}:{PASS}@{HOST}:{PORT}/{NAME}')
print(' Bridge to MySQL ready to use')


 Bridge to MySQL ready to use


In [31]:

def converting_to_csv(df,file_name):
    '''
    Universal function to convert mutiple DataFrame to csv format.
    automaticly detect/make folders, prevent/minimize errors
    '''
    folder_name = 'Analysis Result'
    try:
        if not file_name.endswith('.csv'):
            fle_name = file_name +'.csv'

        os.makedirs(folder_name, exist_ok= True)

        saved_path =os.path.join(folder_name, file_name)
        df.to_csv(saved_path, index= False)

    except Exception as e:
        print(f'failed to convert/save file. Error:{e}')


In [25]:
monthly_query ='''
    select date_format(o.order_purchase_timestamp, '%%Y-%%m') as Month, sum(p.payment_value) as total_revenue
    from orders o join payment p
    on o.order_id = p.order_id
    group by Month 
    order by Month asc
'''
df_Monthly = pd.read_sql(monthly_query, con = engine)
df_Monthly['MoM_growth_percent'] = df_Monthly['total_revenue'].pct_change() * 100 
df_format_Monthly = df_Monthly.style.format({'total_revenue': '{:,.2f}'})
df_format_Monthly


,Month,total_revenue,MoM_growth_percent
0,2016-10,"47,271.20",nan
1,2016-12,19.62,-99.958495
2,2017-01,"127,545.67",649979.867482
3,2017-02,"271,298.65",112.707064
4,2017-03,"414,369.39",52.735515
5,2017-04,"390,952.18",-5.651289
6,2017-05,"566,872.73",44.997971
7,2017-06,"490,225.60",-13.521047
8,2017-07,"566,403.93",15.539443
9,2017-08,"646,000.61",14.052989


In [27]:
df_Monthly.to_csv('Monthly_revenue.csv', index= False, output = 'ready for tableau')

TypeError: NDFrame.to_csv() got an unexpected keyword argument 'output'